# AI 辯論擂台 — 用 AISuite 打造多 LLM 辯論生成器

**AI 辯論擂台** 讓你輸入任何議題，由兩個不同的 LLM 分別擔任「正方」與「反方」辯手，展開多回合激烈辯論，最後再由第三個 LLM 擔任「評審」進行評分與總結！

此版本為**零設定防呆版**，直接使用 Groq 強大的免費開源模型群：

In [ ]:
!pip install aisuite[all] gradio -q

In [ ]:
import aisuite as ai
import os

# 直接寫入你的 Groq API Key（防呆專用）
# Set GROQ_API_KEY in the environment before running this cell.
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError('GROQ_API_KEY is required')

client = ai.Client()

# 為了保證絕對不報錯，全部換成 Groq 目前最穩定、不會下架的模型
model_pro = "groq:llama-3.3-70b-versatile"       # 🔴 正方：Meta LLaMA 3.3 (70B)
model_con = "groq:llama-3.1-8b-instant"          # 🔵 反方：Meta LLaMA 3.1 (8B)
model_judge = "groq:llama-3.3-70b-versatile"     # ⚖️ 評審：Meta LLaMA 3.3 (70B)

print("✅ 環境與 API 鑰匙設定成功！可以換下一格了！")

In [ ]:
# 設定三個不同角色的「人設」
pro_system = """你是一位台灣的專業辯論選手，擔任「正方」立場。請用台灣習慣的繁體中文回覆。
你的任務：堅定支持議題，針對反方反駁，每次發言 150 字以內，語氣要有氣勢。"""

con_system = """你是一位台灣的專業辯論選手，擔任「反方」立場。請用台灣習慣的繁體中文回覆。
你的任務：堅定反對議題，針對正方反駁，每次發言 150 字以內，犀利且具批判性。"""

judge_system = """你是一位經驗豐富的台灣辯論賽評審。請用繁體中文回覆。
請根據邏輯性、說服力、攻防表現進行評分。
格式：
1. 雙方得分 (滿分10分)
2. 簡要評語
3. 宣布最終勝負"""

def get_response(model, messages):
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

print("✅ 人設準備完畢！")

In [ ]:
def run_debate(topic, rounds=2):
    debate_log = [f"🏟️ AI 辯論擂台\n📋 議題：{topic}\n" + "="*40]

    pro_history = [{"role": "system", "content": pro_system}]
    con_history = [{"role": "system", "content": con_system}]
    full_debate = ""
    con_speech = ""

    for r in range(1, rounds + 1):
        debate_log.append(f"\n🔔 === 第 {r} 回合 ===")

        # 正方
        prompt_pro = f"辯論議題：「{topic}」\n發表第一輪立論。" if r == 1 else f"對方反方剛才說：\n{con_speech}\n請反駁對方，加強正方立場。"
        pro_history.append({"role": "user", "content": prompt_pro})
        pro_speech = get_response(model_pro, pro_history)
        pro_history.append({"role": "assistant", "content": pro_speech})
        debate_log.append(f"\n🔴【正方 - LLaMA 3.3】：\n{pro_speech}")
        full_debate += f"\n[正方 第{r}回合]：\n{pro_speech}\n"

        # 反方
        prompt_con = f"辯論議題：「{topic}」\n正方剛才說：\n{pro_speech}\n請發表反對立論並反駁正方。" if r == 1 else f"對方正方剛才說：\n{pro_speech}\n請反駁對方，加強反方立場。"
        con_history.append({"role": "user", "content": prompt_con})
        con_speech = get_response(model_con, con_history)
        con_history.append({"role": "assistant", "content": con_speech})
        debate_log.append(f"\n🔵【反方 - LLaMA 3.1】：\n{con_speech}")
        full_debate += f"\n[反方 第{r}回合]：\n{con_speech}\n"

    # 評審
    debate_log.append(f"\n" + "="*40 + "\n⚖️ === 評審評判===")
    judge_messages = [
        {"role": "system", "content": judge_system},
        {"role": "user", "content": f"這是關於「{topic}」的辯論紀錄：\n{full_debate}\n請公正當裁判。"}
    ]
    judgment = get_response(model_judge, judge_messages)
    debate_log.append(f"\n{judgment}")

    return "\n".join(debate_log)

print("✅ 辯論擂台主程式建立成功！")

In [ ]:
# 測試
print("⚡ 正在呼叫三個 AI 展開辯論，請等待大約 20 秒...")
result = run_debate("小學生應該要有薪水", rounds=1)
print(result)

In [ ]:
import gradio as gr

def debate_interface(topic, rounds):
    if not topic.strip(): return "⚠️ 請輸入辯論議題！"
    return run_debate(topic, rounds=int(rounds))

with gr.Blocks(title="AI 辯論擂台", theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    gr.Markdown("# 🎭 AI 辯論擂台\n輸入議題，讓三個頂尖開源 AI 幫你吵架兼做裁判！")

    with gr.Row():
        topic_input = gr.Textbox(label="🏷️ 辯論議題", placeholder="例如：狗比貓可愛", scale=3)
        rounds_input = gr.Slider(minimum=1, maximum=4, value=2, step=1, label="🔄 回合數", scale=1)

    start_btn = gr.Button("⚔️ 開始辯論！", variant="primary", size="lg")
    output = gr.Textbox(label="📜 即時文字轉播區", lines=20)

    gr.Examples([["吃火鍋必加芋頭", 2], ["世界上有外星人", 2], ["大學生不該用AI寫報告", 2]], inputs=[topic_input, rounds_input])
    start_btn.click(fn=debate_interface, inputs=[topic_input, rounds_input], outputs=output)

demo.launch(share=True, debug=True)